# Task 7.15 — Đóng Gói Mô Hình Hợp Nhất & Kiểm Chuẩn Độ Trễ (Unified Packaging & Latency Benchmark)

> **Mục tiêu**: Thực hiện toàn trình quy trình chuẩn hóa và kiểm định sản phẩm triển khai trên GPU Colab:
> 1. **Trích xuất Logits Thực Nghiệm Độc Lập**: Checkpoint v2 (`fatformer_anchored_v2_best.pth`, SHA-256 `5cd49252...`) chạy forward trên 400 ảnh tập hiệu chuẩn độc lập `val_calib` (200 real, 200 fake).
> 2. **Hiệu Chuẩn Riêng Biệt Cho Checkpoint v2**: Dùng `tools.calibrate_and_select` để fit nhiệt độ tối ưu $T$ và ngưỡng quyết định $T^*$ ($\text{FPR} \le 5\%$), tạo ra `anchored_v2.json` chính chủ với đầy đủ nguồn chứng chỉ thực nghiệm (không tái sử dụng calibration của baseline).
> 3. **Đóng Gói Checkpoint Hợp Nhất Độc Lập Toàn Phần**: Dùng `tools.package_unified_fatformer` nhúng 1.200 tensors (CLIP backbone + 3 adapter blocks + classifier), layout metadata, calibration chính chủ, và router ablation proof vào một artifact duy nhất `unified_fatformer_v2.pth`.
> 4. **Kiểm Thử Standalone Tự Chứa 100%**: Mô hình tự khôi phục toàn bộ trọng số CLIP mà không phụ thuộc file `ViT-L-14.pt` rời bên ngoài.
> 5. **Kiểm Chuẩn Độ Trễ Khách Quan & Nghiệm Thu Cổng Chặn G7.3**: Đo độ trễ qua `tools.infer_unified --benchmark_latency` trên GPU Tesla T4 với $B \in \{1, 8, 32\}$, đồng bộ CUDA nghiêm ngặt, tách biệt thời gian backbone và overhead đóng gói ($\Delta T < 5.0$ ms/ảnh).
> 6. **Sao Lưu Tự Động Lên Google Drive 5TB**: Đồng bộ toàn bộ artifact nghiệm thu (`unified_fatformer_v2.pth`, `metadata.json`, `anchored_v2.json`, `latency_phase7.json`).

> **Môi trường**: Google Colab GPU (Tesla T4 khuyến nghị theo G7.3) + Google Drive 5TB.

In [ ]:
# CELL 1: KẾT NỐI GOOGLE DRIVE & ĐỒNG BỘ MÃ NGUỒN FATFORMER-XLA (ORIGIN/HOANG)
import os, sys, shutil, subprocess
from pathlib import Path
from google.colab import drive

print("[*] Đang kết nối Google Drive 5TB...")
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/Fatformer')
if not DRIVE_ROOT.exists():
    candidates = [Path('/content/drive/MyDrive/FatFormer_Hub'), Path('/content/drive/MyDrive')]
    for c in candidates:
        if c.exists():
            DRIVE_ROOT = c
            break

print(f"[✓] Thư mục Drive chính: {DRIVE_ROOT}")

REPO_DIR = Path('/content/fatformer-xla')
if not (REPO_DIR / '.git').exists():
    print("[*] Clone nhánh hoang từ GitHub...")
    subprocess.run(['git', 'clone', '-b', 'hoang', 'https://github.com/LeeVHoangtk3/fatformer-xla.git', str(REPO_DIR)], check=True)
else:
    print("[*] Cập nhật nhánh hoang mới nhất...")
    subprocess.run(['git', 'fetch', 'origin', 'hoang'], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', 'origin/hoang'], cwd=REPO_DIR, check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))

commit_sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print(f"[✓] Commit hiện tại: {commit_sha}")

In [ ]:
# CELL 2: CÀI ĐẶT THƯ VIỆN & KIỂM TRA PHẦN CỨNG GPU
import torch

print("[*] Đang cài đặt các thư viện cần thiết...")
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ftfy', 'regex', 'tqdm', 'pyyaml', 'timm', 'einops', 'scikit-learn', 'pandas', 'matplotlib', 'Pillow'], check=True)

print("=" * 70)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"[✓] GPU khả dụng: {gpu_name} ({vram_gb:.2f} GiB VRAM)")
    if "T4" in gpu_name:
        print("[✓] Phần cứng chuẩn Tesla T4: Hợp lệ để nghiệm thu chính thức cổng chặn G7.3 (overhead < 5.0 ms/ảnh).")
    else:
        print(f"[i] Lưu ý: Đang chạy trên {gpu_name}. Nghiệm thu chính thức G7.3 theo đề cương quy định trên T4.")
else:
    print("[!] CẢNH BÁO: Không tìm thấy GPU! Kiểm chuẩn độ trễ G7.3 yêu cầu GPU CUDA.")
print("=" * 70)

In [ ]:
# CELL 3: ĐỊNH VỊ CHECKPOINT NGUỒN V2, PRETRAINED CLIP & TẬP DỮ LIỆU HIỆU CHUẨN
import tarfile, zipfile, glob, hashlib

DATA_STAGE = Path('/content/dataset_local')
DATA_STAGE.mkdir(parents=True, exist_ok=True)

def find_file(candidates):
    for c in candidates:
        if not c:
            continue
        matches = glob.glob(str(c), recursive=True)
        if matches:
            return Path(matches[0])
    return None

def sha256_file(path, block_size=65536):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(block_size), b""):
            h.update(block)
    return h.hexdigest()

print("[*] 1. Định vị Checkpoint v2 (Anchored Run 1) mục tiêu...")
ckpt_candidates = [
    DRIVE_ROOT / 'checkpoint/phase7/anchored_v2_run1/fatformer_anchored_v2_best.pth',
    DRIVE_ROOT / 'checkpoint/phase7/ablation_v2_fixed_run1/a4_full/best.pth',
    DRIVE_ROOT / 'checkpoint/phase7/ablation_v2_fixed_run1/a4_full/fatformer_anchored_v2_best.pth',
    DRIVE_ROOT / '**/fatformer_anchored_v2_best.pth',
    DRIVE_ROOT / '**/best.pth'
]
target_ckpt = find_file(ckpt_candidates)
assert target_ckpt and target_ckpt.exists(), f"[!] LỖI CHẶN: Không tìm thấy checkpoint nào trong {ckpt_candidates}!"
ckpt_hash = sha256_file(target_ckpt)
print(f"[✓] Checkpoint v2 nguồn: {target_ckpt}")
print(f"[✓] SHA-256 Checkpoint: {ckpt_hash}")
EXPECTED_PREFIX = "5cd49252"
if ckpt_hash.startswith(EXPECTED_PREFIX):
    print(f"[✓] Khớp mã băm chuẩn của Checkpoint v2 ({EXPECTED_PREFIX}...)!")
else:
    print(f"[i] Cảnh báo: SHA-256 bắt đầu bằng {ckpt_hash[:8]}, khác tiền tố kỳ vọng {EXPECTED_PREFIX}...")

print("\n[*] 2. Định vị Backbone ViT-L/14...")
clip_candidates = [
    DRIVE_ROOT / 'pretrained/ViT-L-14.pt',
    DRIVE_ROOT / '**/ViT-L-14.pt',
    REPO_DIR / 'DATASET/pretrained/ViT-L-14.pt'
]
clip_path = find_file(clip_candidates)
if clip_path and clip_path.exists():
    print(f"[✓] Tìm thấy CLIP Backbone ngoài: {clip_path}")
else:
    print("[i] Không tìm thấy ViT-L-14.pt ngoài đĩa, package sẽ tự trích xuất hoặc nạp standalone.")

print("\n[*] 3. Định vị Artifacts & Phân Vùng Hiệu Chuẩn val_calib...")
art_zip = find_file([DRIVE_ROOT / 'phase7_measured_artifacts.zip', DRIVE_ROOT / '**/phase7_measured_artifacts.zip'])
if art_zip and art_zip.exists():
    print(f"[*] Đang giải nén {art_zip.name} sang {REPO_DIR}...")
    with zipfile.ZipFile(art_zip, 'r') as zf:
        zf.extractall(REPO_DIR)
    print("[✓] Đã giải nén phase7_measured_artifacts.zip thành công.")

calib_split_candidates = [
    REPO_DIR / 'DATASET/logs/phase7/calibration/logits_baseline_clean_calib_split.npz',
    REPO_DIR / 'DATASET/logs/phase7/logits_baseline_clean_calib_split.npz',
    REPO_DIR / 'DATASET/logs/phase7/calib_split.npz'
]
calib_split_file = find_file(calib_split_candidates)

val_candidates = [
    DATA_STAGE / 'val_crossdomain',
    DRIVE_ROOT / 'datasets/val_crossdomain',
    DRIVE_ROOT / '**/val_crossdomain',
    DATA_STAGE / 'cfe'
]
calib_data_dir = find_file(val_candidates)

if not calib_data_dir:
    tar_cfe = find_file([DRIVE_ROOT / 'datasets/test_communityforensics.tar', DRIVE_ROOT / '**/test_communityforensics.tar'])
    if tar_cfe and tar_cfe.exists():
        print(f"[*] Đang giải nén {tar_cfe.name} để trích xuất ảnh hiệu chuẩn...")
        with tarfile.open(tar_cfe, 'r') as tf:
            tf.extractall(DATA_STAGE / 'cfe')
        calib_data_dir = DATA_STAGE / 'cfe'

print(f"[✓] Thư mục dữ liệu hiệu chuẩn: {calib_data_dir}")
print(f"[✓] Tệp chỉ định phân vùng split: {calib_split_file}")

router_params_path = find_file([REPO_DIR / 'DATASET/logs/phase7/router_params.json', DRIVE_ROOT / '**/router_params.json'])
router_ablation_path = find_file([REPO_DIR / 'DATASET/logs/phase7/router_ablation_summary.json', DRIVE_ROOT / '**/router_ablation_summary.json'])
print(f"[i] Router params: {router_params_path}")
print(f"[i] Router ablation proof: {router_ablation_path}")

In [ ]:
# CELL 4: TRÍCH XUẤT LOGITS THỰC NGHIỆM ĐỘC LẬP CHO CHECKPOINT V2 (400 ẢNH VAL_CALIB)
OUT_LOGITS_NPZ = REPO_DIR / 'DATASET/logs/phase7/logits_anchored_v2_clean_calib.npz'
OUT_LOGITS_NPZ.parent.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print(f"TRÍCH XUẤT LOGITS ĐO THẬT TỪ CHECKPOINT V2 SANG: {OUT_LOGITS_NPZ}")
print("=" * 80)

cmd_extract = [
    sys.executable, '-u', '-m', 'tools.extract_calibration_logits',
    '--checkpoint', str(target_ckpt),
    '--output_npz', str(OUT_LOGITS_NPZ),
    '--device', 'cuda' if torch.cuda.is_available() else 'cpu',
    '--batch_size', '32'
]

if calib_data_dir and Path(calib_data_dir).exists():
    cmd_extract.extend(['--data_dir', str(calib_data_dir)])
if calib_split_file and Path(calib_split_file).exists():
    cmd_extract.extend(['--split_file', str(calib_split_file)])
if clip_path and Path(clip_path).exists():
    cmd_extract.extend(['--clip_path', str(clip_path)])

proc_ext = subprocess.Popen(cmd_extract, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc_ext.stdout:
    print(line, end='', flush=True)
proc_ext.wait()

assert proc_ext.returncode == 0, f"[!] LỖI TRÍCH XUẤT: extract_calibration_logits thoát với mã {proc_ext.returncode}!"
assert OUT_LOGITS_NPZ.exists(), f"[!] LỖI: Không tìm thấy tệp {OUT_LOGITS_NPZ}!"

import numpy as np
with np.load(OUT_LOGITS_NPZ, allow_pickle=True) as npz:
    n_samples = len(npz['targets'])
    real_count = int((npz['targets'] == 0).sum())
    fake_count = int((npz['targets'] == 1).sum())
    npz_sha = sha256_file(OUT_LOGITS_NPZ)
    print("\n" + "=" * 70)
    print(f"[✓] Đã tạo thành công Logits NPZ thật: {OUT_LOGITS_NPZ}")
    print(f"[✓] Số mẫu: {n_samples} (Real: {real_count}, Fake: {fake_count})")
    print(f"[✓] Tiền xử lý: {npz.get('preprocessing', 'N/A')}")
    print(f"[✓] SHA-256 Logits NPZ: {npz_sha}")
    print("=" * 70)

In [ ]:
# CELL 5: HIỆU CHUẨN NHIỆT ĐỘ T & NGƯỠNG T* RIÊNG BIỆT CHO CHECKPOINT V2
import json
from IPython.display import display, Image as IPyImage

CALIB_OUT_DIR = REPO_DIR / 'DATASET/logs/phase7/calibration'
CALIB_OUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print(f"FIT HIỆU CHUẨN NHIỆT ĐỘ T & NGƯỠNG T* CHO CHECKPOINT V2 | TARGET FPR = 0.05")
print("=" * 80)

cmd_calib = [
    sys.executable, '-u', '-m', 'tools.calibrate_and_select',
    '--logits_npz', str(OUT_LOGITS_NPZ),
    '--model_name', 'anchored_v2',
    '--checkpoint', str(target_ckpt),
    '--output_dir', str(CALIB_OUT_DIR),
    '--fpr_target', '0.05'
]

proc_cal = subprocess.Popen(cmd_calib, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc_cal.stdout:
    print(line, end='', flush=True)
proc_cal.wait()

assert proc_cal.returncode == 0, f"[!] LỖI HIỆU CHUẨN: calibrate_and_select thoát với mã {proc_cal.returncode}!"

anchored_calib_json = CALIB_OUT_DIR / 'anchored_v2.json'
assert anchored_calib_json.exists(), f"[!] LỖI: Không tìm thấy file {anchored_calib_json}!\""

with open(anchored_calib_json, 'r', encoding='utf-8') as f:
    calib_res = json.load(f)

print("\n" + "=" * 70)
print(f"KẾT QUẢ HIỆU CHUẨN CHÍNH THỨC CHECKPOINT V2 ({calib_res.get('evidence_level')}):")
print(f"• Nhiệt độ T:             {calib_res.get('T')}")
print(f"• Ngưỡng T* (FPR=0.05):   {calib_res.get('T_star_default')}")
print(f"• FPR thực đo:            {calib_res.get('fpr_measured')}")
print(f"• ECE:                    {calib_res.get('ece_before')}% -> {calib_res.get('ece_after')}%")
print(f"• NLL:                    {calib_res.get('nll_before')} -> {calib_res.get('nll_after')}")
print(f"• Real ACC / Fake ACC:    {calib_res.get('real_acc')}% / {calib_res.get('fake_acc')}%")
print(f"• Checkpoint SHA-256:     {calib_res.get('checkpoint', {}).get('sha256')}")
print("=" * 70)

rel_png = CALIB_OUT_DIR / 'anchored_v2_reliability.png'
if rel_png.exists():
    print("\n[*] Biểu đồ Reliability Diagram (Trước & Sau Hiệu Chuẩn):")
    display(IPyImage(str(rel_png)))

In [ ]:
# CELL 6: ĐÓNG GÓI CHECKPOINT HỢP NHẤT TOÀN VẸN (TASK 7.15)
PKG_OUT_PTH = REPO_DIR / 'DATASET/checkpoints/phase7/unified_fatformer_v2.pth'
META_OUT_JSON = REPO_DIR / 'DATASET/checkpoints/phase7/metadata.json'
PKG_OUT_PTH.parent.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print(f"TIẾN HÀNH ĐÓNG GÓI SẢN PHẨM PHÁT HÀNH: {PKG_OUT_PTH}")
print("=" * 80)

cmd_pkg = [
    sys.executable, '-u', '-m', 'tools.package_unified_fatformer',
    '--checkpoint', str(target_ckpt),
    '--output', str(PKG_OUT_PTH),
    '--calibration', str(anchored_calib_json),
    '--metadata_out', str(META_OUT_JSON)
]

if router_params_path and Path(router_params_path).exists():
    cmd_pkg.extend(['--router_params', str(router_params_path)])
if router_ablation_path and Path(router_ablation_path).exists():
    cmd_pkg.extend(['--router_ablation', str(router_ablation_path)])

proc_pkg = subprocess.Popen(cmd_pkg, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc_pkg.stdout:
    print(line, end='', flush=True)
proc_pkg.wait()

assert proc_pkg.returncode == 0, f"[!] LỖI ĐÓNG GÓI: package_unified_fatformer thoát với mã {proc_pkg.returncode}!"
assert PKG_OUT_PTH.exists(), f"[!] LỖI: Không tìm thấy file package {PKG_OUT_PTH}!"

pkg_size_mb = PKG_OUT_PTH.stat().st_size / (1024 * 1024)
pkg_sha = sha256_file(PKG_OUT_PTH)
print("\n" + "=" * 70)
print(f"[✓] ĐÃ ĐÓNG GÓI THÀNH CÔNG: {PKG_OUT_PTH.name}")
print(f"[✓] Dung lượng: {pkg_size_mb:.2f} MB")
print(f"[✓] SHA-256: {pkg_sha}")
print(f"[✓] Metadata độc lập: {META_OUT_JSON}")
print("=" * 70)

In [ ]:
# CELL 7: ĐO ĐỘ TRỄ PHÂN TẦNG (B=1, 8, 32) & NGHIỆM THU CỔNG CHẶN G7.3 TRÊN GPU
LATENCY_OUT_JSON = REPO_DIR / 'DATASET/logs/phase7/latency_phase7.json'
LATENCY_OUT_JSON.parent.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print(f"CHẠY KIỂM CHUẨN ĐỘ TRỄ PHÂN TẦNG TRÊN GPU ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
print("=" * 80)

cmd_lat = [
    sys.executable, '-u', '-m', 'tools.infer_unified',
    '--checkpoint', str(PKG_OUT_PTH),
    '--device', 'cuda' if torch.cuda.is_available() else 'cpu',
    '--benchmark_latency',
    '--latency_out', str(LATENCY_OUT_JSON)
]

proc_lat = subprocess.Popen(cmd_lat, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc_lat.stdout:
    print(line, end='', flush=True)
proc_lat.wait()

assert proc_lat.returncode == 0, f"[!] LỖI ĐO ĐỘ TRỄ: infer_unified thoát với mã {proc_lat.returncode}!"
assert LATENCY_OUT_JSON.exists(), f"[!] LỖI: Không tìm thấy {LATENCY_OUT_JSON}!"

with open(LATENCY_OUT_JSON, 'r', encoding='utf-8') as f:
    lat_res = json.load(f)

print("\n" + "=" * 80)
print("BẢNG TỔNG HỢP ĐỘ TRỄ & HIỆU NĂNG SUY LUẬN (TASK 7.15 / G7.3):")
print("-" * 80)
print(f"{'Batch Size':<12} | {'Backbone (ms/ảnh)':<18} | {'Package (ms/ảnh)':<18} | {'Overhead ΔT (ms)':<16} | {'Throughput (FPS)':<16}")
print("-" * 80)
for b_str, b_data in lat_res.get("batches", {}).items():
    print(f"{b_data['batch_size']:<12} | {b_data['backbone_per_image_ms']:<18.2f} | {b_data['package_per_image_ms']:<18.2f} | {b_data['overhead_delta_ms']:<16.4f} | {b_data['throughput_fps']:<16.1f}")
print("-" * 80)

meets_g73 = lat_res.get("overhead_meets_target")
b1_overhead = lat_res.get("batches", {}).get("1", {}).get("overhead_delta_ms")
print(f"• Cổng chặn G7.3 (Overhead ΔT batch 1 < 5.0 ms/ảnh):")
if meets_g73 is True:
    print(f"  [✓] ĐẠT (PASS): ΔT = {b1_overhead:.4f} ms < 5.0 ms!")
elif meets_g73 is False:
    print(f"  [✗] KHÔNG ĐẠT (FAIL): ΔT = {b1_overhead:.4f} ms >= 5.0 ms!")
else:
    print(f"  [i] CHƯA ĐÁNH GIÁ (Chạy trên CPU, yêu cầu GPU Tesla T4 để nghiệm thu chính thức).")
print("=" * 80)

In [ ]:
# CELL 8: KIỂM THỬ SUY LUẬN ĐỘC LẬP (STANDALONE) & ĐỒNG BỘ ARTIFACTS SANG GOOGLE DRIVE
from google.colab import files

print("=" * 80)
print("1. KIỂM THỬ SUY LUẬN ĐỘC LẬP KHÔNG PHỤ THUỘC FILE CLIP NGOÀI...")
print("=" * 80)

demo_out_json = REPO_DIR / 'DATASET/logs/phase7/demo_inference_output.json'
cmd_demo = [
    sys.executable, '-u', '-m', 'tools.infer_unified',
    '--checkpoint', str(PKG_OUT_PTH),
    '--device', 'cuda' if torch.cuda.is_available() else 'cpu',
    '--clip_path', 'NON_EXISTENT_DUMMY_CLIP_PATH.pt',
    '--output', str(demo_out_json)
]
if calib_data_dir and Path(calib_data_dir).exists():
    sample_imgs = list(Path(calib_data_dir).rglob('*.png'))[:4] + list(Path(calib_data_dir).rglob('*.jpg'))[:4]
    if sample_imgs:
        cmd_demo.extend(['--images'] + [str(p) for p in sample_imgs[:4]])

subprocess.run(cmd_demo, check=True)
if demo_out_json.exists():
    with open(demo_out_json, 'r', encoding='utf-8') as f:
        demo_res = json.load(f)
    print(f"[✓] Đã chạy suy luận thành công {len(demo_res)} ảnh trong chế độ Standalone:")
    for it in demo_res[:4]:
        print(f"  • {Path(it['image_path']).name}: {it['prediction_label']} (Score: {it['decision_score']:.4f}, Raw Prob: {it['calibrated_probability_fake']:.4f})")

print("\n" + "=" * 80)
print("2. ĐỒNG BỘ TOÀN BỘ ARTIFACTS TASK 7.15 SANG GOOGLE DRIVE 5TB...")
print("=" * 80)

DRIVE_CKPT_DIR = DRIVE_ROOT / 'checkpoint/phase7'
DRIVE_LOGS_DIR = DRIVE_ROOT / 'logs/phase7'
DRIVE_CALIB_DIR = DRIVE_LOGS_DIR / 'calibration'

DRIVE_CKPT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_LOGS_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_CALIB_DIR.mkdir(parents=True, exist_ok=True)

sync_items = [
    (PKG_OUT_PTH, DRIVE_CKPT_DIR / PKG_OUT_PTH.name),
    (META_OUT_JSON, DRIVE_CKPT_DIR / META_OUT_JSON.name),
    (anchored_calib_json, DRIVE_CALIB_DIR / anchored_calib_json.name),
    (LATENCY_OUT_JSON, DRIVE_LOGS_DIR / LATENCY_OUT_JSON.name),
    (OUT_LOGITS_NPZ, DRIVE_LOGS_DIR / OUT_LOGITS_NPZ.name),
    (CALIB_OUT_DIR / 'anchored_v2_reliability.png', DRIVE_CALIB_DIR / 'anchored_v2_reliability.png')
]

for src, dst in sync_items:
    if src.exists():
        shutil.copy2(src, dst)
        print(f"[✓] Đã sao lưu sang Drive: {dst}")

TASK_ZIP_LOCAL = Path('/content/task_7.15_unified_package_artifacts.zip')
with zipfile.ZipFile(TASK_ZIP_LOCAL, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for src, dst in sync_items:
        if src.exists():
            zf.write(src, arcname=f"task_7.15/{src.name}")

drive_zip_target = DRIVE_LOGS_DIR / TASK_ZIP_LOCAL.name
shutil.copy2(TASK_ZIP_LOCAL, drive_zip_target)
print(f"\n[✓] ĐÃ LƯU GÓI PHÁT HÀNH ZIP VÀO DRIVE: {drive_zip_target}")

print("\n[*] Tự động tải file zip về trình duyệt (nếu hỗ trợ)...")
try:
    files.download(str(TASK_ZIP_LOCAL))
    print("[✓] Đã gửi lệnh tải về trình duyệt.")
except Exception as e:
    print(f"[i] Không thể tự tải trình duyệt: {e}. Vui lòng lấy trực tiếp trên Drive: {drive_zip_target}")